# 00 — Recopilación de datos (Fase 1)

Este notebook reproduce íntegramente la descarga y generación de todos los datasets de `data/raw/`.  
Ejecutar de arriba a abajo re-descarga los datos desde sus fuentes originales.

| Dataset | Fuente | Licencia |
|---|---|---|
| `results.csv` | [martj42/international_results](https://github.com/martj42/international_results) | CC0 |
| `goalscorers.csv` | martj42/international_results | CC0 |
| `shootouts.csv` | martj42/international_results | CC0 |
| `elo_ratings.csv` | [eloratings.net](https://www.eloratings.net/) | Educativo |
| `fifa_ranking.csv` | [inside.fifa.com API](https://inside.fifa.com/fifa-world-ranking/men) | Público |
| `wc2026_groups.csv` | Sorteo oficial FIFA dic-2025 (manual) | Dominio público |

## 0. Imports y configuración

In [1]:
import requests
import pandas as pd
import time
from pathlib import Path

RAW = Path('../data/raw')
RAW.mkdir(parents=True, exist_ok=True)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'
}

print(f'Carpeta de destino: {RAW.resolve()}')

Carpeta de destino: /Users/alejandrogomeznavas/Desktop/Claude-Work/Proyectos/Predicción Mundial/data/raw


---
## 1. Partidos internacionales — martj42/international_results

Dataset de dominio público (CC0) con todos los partidos internacionales desde 1872.  
Incluye los 72 partidos de fase de grupos del Mundial 2026 (resultados pendientes = NaN).

In [2]:
MARTJ42_BASE = 'https://raw.githubusercontent.com/martj42/international_results/master'

files = {
    'results.csv':     f'{MARTJ42_BASE}/results.csv',
    'goalscorers.csv': f'{MARTJ42_BASE}/goalscorers.csv',
    'shootouts.csv':   f'{MARTJ42_BASE}/shootouts.csv',
}

for filename, url in files.items():
    dest = RAW / filename
    response = requests.get(url, headers=HEADERS, timeout=60)
    response.raise_for_status()
    dest.write_bytes(response.content)
    size_kb = dest.stat().st_size / 1024
    print(f'✓ {filename:<22} {size_kb:>8.1f} KB')

✓ results.csv              3630.7 KB


✓ goalscorers.csv          3180.4 KB


✓ shootouts.csv              28.1 KB


In [3]:
results = pd.read_csv(RAW / 'results.csv')

print(f'Partidos totales : {len(results):,}')
print(f'Rango de fechas  : {results.date.min()} → {results.date.max()}')
print(f'Columnas         : {list(results.columns)}')
print()

# Partidos del Mundial 2026 ya en el dataset
wc2026 = results[
    (results.tournament == 'FIFA World Cup') & (results.date >= '2026-01-01')
]
print(f'Partidos Mundial 2026 en el dataset: {len(wc2026)}')
results.tail(3)

Partidos totales : 49,328
Rango de fechas  : 1872-11-30 → 2026-06-27
Columnas         : ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']

Partidos Mundial 2026 en el dataset: 72


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49325,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49326,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True
49327,2026-06-27,Croatia,Ghana,NaN,NaN,FIFA World Cup,Philadelphia,United States,True


In [4]:
print('Top 10 torneos por número de partidos:')
results['tournament'].value_counts().head(10)

Top 10 torneos por número de partidos:


tournament
Friendly                                18256
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
Name: count, dtype: int64

---
## 2. Elo ratings — eloratings.net

El sitio sirve los datos como archivos `.tsv` sin robots.txt restrictivo.  
Descargamos el ranking mundial actual (`World.tsv`) y el mapeo de códigos a nombres (`en.teams.tsv`).

> **Nota:** Este dataset es un **snapshot estático** (fecha de descarga).  
> El Elo histórico por partido se calculará en Fase 3 desde `results.csv`.

In [5]:
ELO_BASE = 'https://www.eloratings.net'

# Mapeo código interno → nombre del equipo
teams_raw = requests.get(f'{ELO_BASE}/en.teams.tsv', headers=HEADERS, timeout=30).text
code_to_name = {}
for line in teams_raw.strip().split('\n'):
    parts = line.strip().split('\t')
    # Ignorar entradas de localización (sufijo _loc)
    if len(parts) >= 2 and not parts[0].endswith('_loc'):
        code_to_name[parts[0]] = parts[1]

print(f'Equipos en el mapeo: {len(code_to_name)}')

Equipos en el mapeo: 304


In [6]:
# Ranking mundial actual
# Columnas relevantes del TSV:
#   col 0 → rank global
#   col 2 → código de equipo
#   col 3 → Elo actual
#   col 5 → Elo máximo histórico
world_raw = requests.get(f'{ELO_BASE}/World.tsv', headers=HEADERS, timeout=30).text

rows = []
for line in world_raw.strip().split('\n'):
    parts = line.strip().split('\t')
    if len(parts) >= 6:
        def clean(v): return v.replace('\u2212', '-').replace('+', '')
        rows.append({
            'rank':       int(parts[0]),
            'code':       parts[2],
            'team':       code_to_name.get(parts[2], f'[{parts[2]}]'),
            'elo_rating': int(clean(parts[3])),
            'peak_elo':   int(clean(parts[5])),
        })

elo_df = pd.DataFrame(rows).sort_values('rank').reset_index(drop=True)
elo_df.to_csv(RAW / 'elo_ratings.csv', index=False)

print(f'✓ elo_ratings.csv — {len(elo_df)} selecciones')
elo_df.head(10)

✓ elo_ratings.csv — 244 selecciones


,rank,code,team,elo_rating,peak_elo
0,1,ES,Spain,2165,2189
1,2,AR,Argentina,2113,2172
2,3,FR,France,2082,2135
3,4,EN,England,2020,2213
4,5,BR,Brazil,1984,2195
5,5,PT,Portugal,1984,2060
6,7,CO,Colombia,1975,2069
7,8,NL,Netherlands,1961,2153
8,9,EC,Ecuador,1933,1934
9,10,HR,Croatia,1930,2015


---
## 3. FIFA World Ranking histórico — inside.fifa.com API

La API pública de FIFA expone rankings por `dateId`. Los IDs válidos se extraen directamente del HTML de la página de rankings.

Cobertura obtenida: **sept 2020 → sept 2025** (36 publicaciones, ~211 equipos por corte).  
La API no expone publicaciones posteriores a sept 2025 en este momento.

In [7]:
import re

FIFA_RANKING_PAGE = 'https://inside.fifa.com/fifa-world-ranking/men'
FIFA_API_BASE     = 'https://inside.fifa.com/api/ranking-overview?locale=en&dateId=id{}'

# Extraer todos los IDs de publicación del HTML de la página
page_html = requests.get(FIFA_RANKING_PAGE, headers=HEADERS, timeout=15).text
raw_ids = re.findall(r'"id(\d{4,5})"', page_html)
ranking_ids = sorted(set(int(x) for x in raw_ids if 13000 <= int(x) <= 20000))

print(f'IDs de publicación encontrados: {len(ranking_ids)}')
print(f'Rango: {min(ranking_ids)} → {max(ranking_ids)}')

IDs de publicación encontrados: 36
Rango: 13043 → 14870


In [8]:
all_rows = []

for did in ranking_ids:
    try:
        r = requests.get(FIFA_API_BASE.format(did), headers=HEADERS, timeout=10)
        data = r.json()
        rankings = data.get('rankings', [])
        if not rankings:
            continue

        pub_date = rankings[0].get('lastUpdateDate', '')[:10]

        for entry in rankings:
            item = entry.get('rankingItem', {})
            all_rows.append({
                'date_id':      did,
                'ranking_date': pub_date,
                'rank':         item.get('rank'),
                'team':         item.get('name'),
                'country_code': item.get('countryCode'),
                'points':       item.get('totalPoints'),
                'confederation': entry.get('tag', {}).get('id'),
            })

        print(f'  ID {did} | {pub_date} | {len(rankings)} equipos')
        time.sleep(0.1)  # ser amables con el servidor

    except Exception as e:
        print(f'  ✗ ID {did} | Error: {e}')

fifa_df = pd.DataFrame(all_rows)
fifa_df.to_csv(RAW / 'fifa_ranking.csv', index=False)

print(f'\n✓ fifa_ranking.csv')
print(f'  Publicaciones : {fifa_df.ranking_date.nunique()}')
print(f'  Filas totales : {len(fifa_df):,}')
print(f'  Fechas        : {fifa_df.ranking_date.min()} → {fifa_df.ranking_date.max()}')

  ID 13043 | 2020-09-17 | 210 equipos


  ID 13078 | 2020-10-22 | 210 equipos


  ID 13113 | 2020-11-27 | 210 equipos


  ID 13127 | 2020-12-10 | 210 equipos


  ID 13197 | 2021-02-18 | 210 equipos


...

  ID 14576 | 2024-11-28 | 211 equipos


  ID 14597 | 2024-12-19 | 211 equipos


  ID 14702 | 2025-04-03 | 211 equipos


  ID 14800 | 2025-07-10 | 211 equipos


  ID 14870 | 2025-09-18 | 211 equipos

✓ fifa_ranking.csv
  Publicaciones : 36
  Filas totales : 7,576
  Fechas        : 2020-09-17 → 2025-09-18


In [9]:
print('Top 10 del ranking más reciente:')
latest_date = fifa_df.ranking_date.max()
(
    fifa_df[fifa_df.ranking_date == latest_date]
    .sort_values('rank')
    .head(10)
    [['rank', 'team', 'country_code', 'points', 'confederation']]
    .reset_index(drop=True)
)

Top 10 del ranking más reciente:


,rank,team,country_code,points,confederation
0,1.0,Spain,ESP,1875.37,UEFA
1,2.0,France,FRA,1870.92,UEFA
2,3.0,Argentina,ARG,1870.32,CONMEBOL
3,4.0,England,ENG,1820.44,UEFA
4,5.0,Portugal,POR,1779.55,UEFA
5,6.0,Brazil,BRA,1761.60,CONMEBOL
6,7.0,Netherlands,NED,1754.17,UEFA
7,8.0,Belgium,BEL,1739.54,UEFA
8,9.0,Croatia,CRO,1714.20,UEFA
9,10.0,Italy,ITA,1710.06,UEFA


---
## 4. Grupos del Mundial 2026

Basado en el sorteo oficial de la FIFA (diciembre 2025).  
Los nombres de los equipos están normalizados para coincidir con `results.csv`.

In [10]:
groups_data = [
    # group, team, confederation, host_country, pot
    ('A', 'Mexico',                 'CONCACAF',  'Yes', 1),
    ('A', 'South Korea',            'AFC',       'No',  2),
    ('A', 'South Africa',           'CAF',       'No',  3),
    ('A', 'Czech Republic',         'UEFA',      'No',  4),
    ('B', 'Canada',                 'CONCACAF',  'Yes', 1),
    ('B', 'Switzerland',            'UEFA',      'No',  2),
    ('B', 'Bosnia and Herzegovina', 'UEFA',      'No',  3),
    ('B', 'Qatar',                  'AFC',       'No',  4),
    ('C', 'United States',          'CONCACAF',  'Yes', 1),
    ('C', 'Turkey',                 'UEFA',      'No',  2),
    ('C', 'Australia',              'AFC',       'No',  3),
    ('C', 'Paraguay',               'CONMEBOL',  'No',  4),
    ('D', 'Germany',                'UEFA',      'No',  1),
    ('D', 'Ecuador',                'CONMEBOL',  'No',  2),
    ('D', 'Ivory Coast',            'CAF',       'No',  3),
    ('D', 'Curaçao',                'CONCACAF',  'No',  4),
    ('E', 'Japan',                  'AFC',       'No',  1),
    ('E', 'Netherlands',            'UEFA',      'No',  2),
    ('E', 'Sweden',                 'UEFA',      'No',  3),
    ('E', 'Tunisia',                'CAF',       'No',  4),
    ('F', 'Spain',                  'UEFA',      'No',  1),
    ('F', 'Saudi Arabia',           'AFC',       'No',  2),
    ('F', 'Uruguay',                'CONMEBOL',  'No',  3),
    ('F', 'Cape Verde',             'CAF',       'No',  4),
    ('G', 'France',                 'UEFA',      'No',  1),
    ('G', 'Senegal',                'CAF',       'No',  2),
    ('G', 'Iraq',                   'AFC',       'No',  3),
    ('G', 'Norway',                 'UEFA',      'No',  4),
    ('H', 'Brazil',                 'CONMEBOL',  'No',  1),
    ('H', 'Morocco',                'CAF',       'No',  2),
    ('H', 'Scotland',               'UEFA',      'No',  3),
    ('H', 'Haiti',                  'CONCACAF',  'No',  4),
    ('I', 'Portugal',               'UEFA',      'No',  1),
    ('I', 'Colombia',               'CONMEBOL',  'No',  2),
    ('I', 'Uzbekistan',             'AFC',       'No',  3),
    ('I', 'DR Congo',               'CAF',       'No',  4),
    ('J', 'England',                'UEFA',      'No',  1),
    ('J', 'Panama',                 'CONCACAF',  'No',  2),
    ('J', 'Croatia',                'UEFA',      'No',  3),
    ('J', 'Ghana',                  'CAF',       'No',  4),
    ('K', 'Argentina',              'CONMEBOL',  'No',  1),
    ('K', 'Algeria',                'CAF',       'No',  2),
    ('K', 'Austria',                'UEFA',      'No',  3),
    ('K', 'Jordan',                 'AFC',       'No',  4),
    ('L', 'Belgium',                'UEFA',      'No',  1),
    ('L', 'Iran',                   'AFC',       'No',  2),
    ('L', 'New Zealand',            'OFC',       'No',  3),
    ('L', 'Egypt',                  'CAF',       'No',  4),
]

groups_df = pd.DataFrame(groups_data, columns=['group', 'team', 'confederation', 'host_country', 'pot'])
groups_df.to_csv(RAW / 'wc2026_groups.csv', index=False)

print(f'✓ wc2026_groups.csv — {len(groups_df)} equipos en {groups_df.group.nunique()} grupos')
print()
print('Equipos por confederación:')
print(groups_df.groupby('confederation')['team'].count().sort_values(ascending=False).to_string())

✓ wc2026_groups.csv — 48 equipos en 12 grupos

Equipos por confederación:
confederation
UEFA        16
CAF         10
AFC          9
CONCACAF     6
CONMEBOL     6
OFC          1


In [11]:
print('Grupos completos:')
groups_df.groupby('group')['team'].apply(list)

Grupos completos:


group
A    [Mexico, South Korea, South Africa, Czech Repu...
B    [Canada, Switzerland, Bosnia and Herzegovina, ...
C         [United States, Turkey, Australia, Paraguay]
D             [Germany, Ecuador, Ivory Coast, Curaçao]
E                [Japan, Netherlands, Sweden, Tunisia]
F           [Spain, Saudi Arabia, Uruguay, Cape Verde]
G                      [France, Senegal, Iraq, Norway]
H                   [Brazil, Morocco, Scotland, Haiti]
I           [Portugal, Colombia, Uzbekistan, DR Congo]
J                    [England, Panama, Croatia, Ghana]
K                [Argentina, Algeria, Austria, Jordan]
L                  [Belgium, Iran, New Zealand, Egypt]
Name: team, dtype: object

---
## 5. Resumen final — verificación de data/raw/

In [ ]:
import os

print('Estado final de data/raw/\n')
print(f'{"Archivo":<25} {"Filas":>8}  {"Tamaño":>10}')
print('-' * 50)

summary = [
    ('results.csv',        RAW / 'results.csv'),
    ('goalscorers.csv',    RAW / 'goalscorers.csv'),
    ('shootouts.csv',      RAW / 'shootouts.csv'),
    ('elo_ratings.csv',    RAW / 'elo_ratings.csv'),
    ('fifa_ranking.csv',   RAW / 'fifa_ranking.csv'),
    ('wc2026_groups.csv',  RAW / 'wc2026_groups.csv'),
]

for name, path in summary:
    if path.exists():
        df = pd.read_csv(path)
        size = path.stat().st_size
        size_str = f'{size/1024:.1f} KB' if size < 1e6 else f'{size/1e6:.1f} MB'
        print(f'{name:<25} {len(df):>8,}  {size_str:>10}')
    else:
        print(f'{name:<25} {"FALTANTE":>8}')

Estado final de data/raw/

Archivo                      Filas      Tamaño
--------------------------------------------------
results.csv                 49,328      3.7 MB


goalscorers.csv             47,601      3.3 MB
shootouts.csv                  677     28.1 KB
elo_ratings.csv                244      6.3 KB
fifa_ranking.csv             7,576    363.4 KB
wc2026_groups.csv               48      1.1 KB

✓ Fase 1 completada. Siguiente paso: notebooks/01_EDA.ipynb
